In [1]:
import numpy as np
import pandas as pd
import pickle

from tensorflow.keras.models import load_model

I0000 00:00:1790355535.621070   17021 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1790355537.453303   17021 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1790355543.399186   17021 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [ ]:
model = load_model("model.keras")
with open("scaler.pkl", "rb") as file:
    scaler = pickle.load(file)

In [ ]:
df = pd.read_csv("Dataset/PRSA_Data_Guanyuan_20130301-20170228.csv")
df["datetime"] = pd.to_datetime(df[["year", "month", "day", "hour"]])
df = df[["datetime", "PM2.5"]]
df = df.set_index("datetime")
df = df.sort_index()
df["PM2.5"] = pd.to_numeric(df["PM2.5"], errors="coerce")
df["PM2.5"] = df["PM2.5"].interpolate(method="time")
df = df.dropna()
df.head()

,PM2.5
datetime,
2013-03-01 00:00:00,4.0
2013-03-01 01:00:00,4.0
2013-03-01 02:00:00,3.0
2013-03-01 03:00:00,3.0
2013-03-01 04:00:00,3.0


In [ ]:
SEQUENCE_LENGTH = 24
latest_data = df["PM2.5"].values[-SEQUENCE_LENGTH:]
print("Number of observations:", len(latest_data))

Number of observations: 24


In [ ]:
latest_scaled = scaler.transform(latest_data.reshape(-1, 1))

In [ ]:
X_input = latest_scaled.reshape(1, SEQUENCE_LENGTH, 1)
print("Input Shape:", X_input.shape)

Input Shape: (1, 24, 1)


In [10]:
prediction_scaled = model.predict(X_input)
prediction = scaler.inverse_transform(prediction_scaled)
predicted_pm25 = prediction[0][0]
print(f"Predicted next-hour PM2.5: " f"{predicted_pm25:.2f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 623ms/step
Predicted next-hour PM2.5: 21.36
